<a href="https://colab.research.google.com/github/siennaLbertsch/Stock-Profile-Analyzer/blob/main/DRAFT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import pandas as pd
import yfinance as yf
import requests

from io import StringIO


class Node:
    def __init__(self, date, price):
        self.date = date
        self.price = price
        self.left = None
        self.right = None

class BST:
    def __init__(self):
        self.root = None

    def insert(self, date, price):
        self.root = self.insert_node(self.root, date, price)

    def insert_node(self, node, date, price):
        if node == None:
            return Node(date, price)
        if date < node.date:
            node.left = self.insert_node(node.left, date, price)
        elif date > node.date:
            node.right = self.insert_node(node.right, date, price)
        else:
            node.price = price
        return node

    def search(self, date):
        return self._search(self.root, date)

    def _search(self, node, date):
        if node == None or node.date == date:
            return node
        if date < node.date:
            return self._search(node.left, date)
        return self._search(node.right, date)

    def inorder(self):
        result = []
        self.inorder_helper(self.root, result)
        return result

    def inorder_helper(self, node, result):
        if node:
            self.inorder_helper(node.left, result)
            result.append((node.date, node.price))
            self.inorder_helper(node.right, result)

    def range_query(self, start_date, end_date):
        result = []
        self.range_query_helper(self.root, start_date, end_date, result)
        return result

    def range_query_helper(self, node, start, end, result):
        if node:
            if start <= node.date <= end:
                result.append((node.date, node.price))
            if node.date > start:
                self.range_query_helper(node.left, start, end, result)
            if node.date < end:
                self.range_query_helper(node.right, start, end, result)


class Stock:
    def __init__(self, name, ticker):
        self.name = name
        self.ticker = ticker
        self.history = BST()

    def addrecord(self, date, price):
        self.history.insert(date, price)

    def getprice(self, date):
        node = self.history.search(date)
        if node:
            return node.price
        return None

    def getrange(self, start, end):
        return self.history.range_query(start, end)

    def trend(self):
        records = self.history.inorder()
        if len(records) == 0:
            return None
        start_price = records[0][1]
        end_price = records[-1][1]
        change = ((end_price - start_price) / start_price) * 100
        return round(change, 2)


stock_table = [[] for _ in range(10)]

def hash_function(ticker):
    total = 0
    for char in ticker:
        total += ord(char)
    return total % 10

def add_stock(ticker, stock):
    bucket = hash_function(ticker)
    stock_table[bucket].append((ticker, stock))
    push_action(("add", ticker, stock))

def get_stock(ticker):
    bucket = hash_function(ticker)
    for item in stock_table[bucket]:
        if item[0] == ticker:
            return item[1]
    return "Not found"

def update_stock(ticker, new_stock):
    bucket = hash_function(ticker)
    for i, item in enumerate(stock_table[bucket]):
        if item[0] == ticker:
            old_stock = stock_table[bucket][i][1]
            stock_table[bucket][i] = (ticker, new_stock)
            push_action(("update", ticker, old_stock))
            return
    return "Not found"

def remove_stock(ticker):
    bucket = hash_function(ticker)
    for i, item in enumerate(stock_table[bucket]):
        if item[0] == ticker:
            removed_stock = stock_table[bucket][i][1]
            stock_table[bucket].pop(i)
            push_action(("remove", ticker, removed_stock))
            return
    return "Not found"


action_stack = []

def push_action(action):
    action_stack.append(action)

def undo_last():
    if len(action_stack) == 0:
        print("Nothing to undo")
        return

    action = action_stack.pop()

    if action[0] == "add":
        ticker = action[1]
        bucket = hash_function(ticker)
        for i, item in enumerate(stock_table[bucket]):
            if item[0] == ticker:
                stock_table[bucket].pop(i)
                print("Undid add: removed " + ticker)
                return

    elif action[0] == "remove":
        ticker = action[1]
        old_stock = action[2]
        bucket = hash_function(ticker)
        stock_table[bucket].append((ticker, old_stock))
        print("Undid remove: restored " + ticker)

    elif action[0] == "update":
        ticker = action[1]
        old_stock = action[2]
        bucket = hash_function(ticker)
        for i, item in enumerate(stock_table[bucket]):
            if item[0] == ticker:
                stock_table[bucket][i] = (ticker, old_stock)
                print("Undid update: reverted " + ticker)
                return


def get_sp500_metadata():
    url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
    response = requests.get(url, headers={"User-Agent": "Mozilla/5.0"})
    all_tables = pd.read_html(StringIO(response.text))

    for table in all_tables:
        if "Symbol" in table.columns and "Security" in table.columns:
            sp500_table = table
            break

    meta = sp500_table[["Symbol", "Security"]].copy()
    meta.columns = ["Ticker", "Company"]
    meta["Ticker"] = meta["Ticker"].str.replace(".", "-", regex=False)

    return meta


def download_returns(meta):
    tickers = meta["Ticker"].tolist()
    data = yf.download(tickers, period="2y", interval="1d", auto_adjust=True, progress=False)

    prices = data["Close"].sort_index()
    prices = prices.dropna(axis=1, how="all")

    valid_tickers = prices.columns.tolist()
    meta = meta[meta["Ticker"].isin(valid_tickers)].reset_index(drop=True)

    returns = prices.pct_change().dropna(how="all")

    return meta, prices, returns


def main():
    print("Fetching S&P 500 company list...")
    meta = get_sp500_metadata()

    print("Downloading prices for " + str(len(meta)) + " stocks...")
    meta, prices, returns = download_returns(meta)

    print("Building stock objects...")
    for ticker in prices.columns:
        company_name = meta[meta["Ticker"] == ticker]["Company"].values[0]
        stock = Stock(company_name, ticker)
        for date, price in prices[ticker].dropna().items():
            stock.addrecord(str(date.date()), price)
        add_stock(ticker, stock)

    meta.to_csv("sp500_metadata.csv", index=False)
    prices.to_csv("sp500_prices.csv")
    returns.to_csv("sp500_returns.csv")

    print("Done! " + str(len(meta)) + " stocks loaded.")

    ticker = input("\nEnter a ticker to look up (e.g. AAPL): ")
    stock = get_stock(ticker)
    if stock != "Not found":
        print("  Trend over 2 years: " + str(stock.trend()) + "%")
        date = input("Enter a date to look up (YYYY-MM-DD): ")
        print("  Price on " + date + ": " + str(stock.getprice(date)))
        start = input("Enter a start date for range (YYYY-MM-DD): ")
        end = input("Enter an end date for range (YYYY-MM-DD): ")
        print("  Prices from " + start + " to " + end + ": " + str(stock.getrange(start, end)))
    else:
        print(ticker + " not found in hash table")

    print("\nTesting undo:")
    remove_stock(ticker)
    print("  After remove, " + ticker + ": " + str(get_stock(ticker)))
    undo_last()
    print("  After undo, " + ticker + " trend: " + str(get_stock(ticker).trend()) + "%")


main()

Fetching S&P 500 company list...


/tmp/ipykernel_3781/1503155539.py:204: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  returns = prices.pct_change().dropna(how="all")


Building stock objects...
Done! 503 stocks loaded.

Enter a ticker to look up (e.g. AAPL): AAPL
  Trend over 2 years: 56.67%
Enter a date to look up (YYYY-MM-DD): 2025-04-08
  Price on 2025-04-08: 171.6717987060547
Enter a start date for range (YYYY-MM-DD): 2025-04-08
Enter an end date for range (YYYY-MM-DD): 2026-04-08
  Prices from 2025-04-08 to 2026-04-08: [('2025-04-08', 171.6717987060547), ('2025-04-09', 197.98709106445312), ('2025-04-10', 189.5936737060547), ('2025-04-11', 197.29013061523438), ('2025-04-14', 201.6411590576172), ('2025-04-15', 201.2628173828125), ('2025-04-16', 193.4269561767578), ('2025-04-17', 196.12521362304688), ('2025-04-21', 192.32177734375), ('2025-04-22', 198.87322998046875), ('2025-04-23', 203.71212768554688), ('2025-04-24', 207.46575927734375), ('2025-04-25', 208.371826171875), ('2025-04-28', 209.22808837890625), ('2025-04-29', 210.29347229003906), ('2025-04-30', 211.57785034179688), ('2025-05-01', 212.39430236816406), ('2025-05-02', 204.45889282226562),